In [1]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score
from tqdm import tqdm



In [ ]:
#helper codes
def onehot_encode(seq, max_len):
    mat = np.zeros((20, max_len), dtype=np.float32)
    for i, aa in enumerate(seq[:max_len]):
        if aa in AA_TO_INDEX:
            mat[AA_TO_INDEX[aa], i] = 1.0
    return mat

In [ ]:
# ------------------------- Helper -------------------------
AA_TO_INDEX = {aa: idx for idx, aa in enumerate("ACDEFGHIKLMNPQRSTVWY")}

class ProteinDataset(Dataset):
    def __init__(self, sequences, labels, max_len):
        self.sequences = sequences
        self.labels = labels
        self.max_len = max_len

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        seq = self.sequences[idx]
        label = self.labels[idx]
        onehot = onehot_encode(seq, self.max_len)
        return torch.tensor(onehot), torch.tensor(label, dtype=torch.float32)


In [ ]:
# ------------------------- Transformer Model -------------------------
class ProteinTransformer(nn.Module):
    def __init__(self, input_dim=20, max_len=512, d_model=128, nhead=4, num_layers=2):
        super(ProteinTransformer, self).__init__()
        self.embedding = nn.Linear(input_dim, d_model)
        self.positional_encoding = self._generate_positional_encoding(d_model, max_len)

        # encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=256)
        encoder_layer = nn.TransformerEncoderLayer(d_model=d_model, nhead=nhead, dim_feedforward=256, batch_first=True)

        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool1d(1),
            nn.Flatten(),
            nn.Linear(d_model, 1),
            nn.Sigmoid()
        )

    def _generate_positional_encoding(self, d_model, max_len):
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        return pe.unsqueeze(0).transpose(1, 2)

    # def forward(self, x):
    #     x = x.permute(0, 2, 1)
    #     x = self.embedding(x)
    #     pe = self.positional_encoding[:, :, :x.size(1)].to(x.device)
    #     x = x + pe.permute(0, 2, 1)
    #     x = x.permute(1, 0, 2)
    #     x = self.transformer_encoder(x)
    #     x = x.permute(1, 2, 0)
    #     return self.classifier(x).squeeze()

    def forward(self, x):
        x = x.permute(0, 2, 1)  # [B, L, C], input is [B, C, L]
        x = self.embedding(x)   # [B, L, d_model]

        pe = self.positional_encoding[:, :, :x.size(1)].to(x.device)  # [1, d_model, L]
        x = x + pe.permute(0, 2, 1)  # [1, L, d_model]

        x = self.transformer_encoder(x)  # [B, L, d_model]
        x = x.permute(0, 2, 1)  # [B, d_model, L] for pooling
        return self.classifier(x).squeeze()


In [ ]:
# ------------------------- Training & Evaluation -------------------------
def train_transformer_model(gene_name, df, save_dir="trained_transformer_models"):
    os.makedirs(save_dir, exist_ok=True)
    df = df[df["Frameshift_Mutation"] == 0].copy()
    df = df[df["Phenotype"].isin(["R", "S"])]
    df["label"] = df["Phenotype"].map({"S": 0, "R": 1})

    sequences = df["Protein_Sequence"].tolist()
    labels = df["label"].values
    max_len = max(len(seq) for seq in sequences)

    dataset = ProteinDataset(sequences, labels, max_len)
    dataloader = DataLoader(dataset, batch_size=64, shuffle=True)

    model = ProteinTransformer(max_len=max_len)
    model = model.cuda()
    criterion = nn.BCELoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-4)

    model.train()
    for epoch in range(10):
        losses, all_preds, all_labels = [], [], []
        for X, y in tqdm(dataloader):
            X, y = X.cuda(), y.cuda()
            optimizer.zero_grad()
            outputs = model(X)
            loss = criterion(outputs, y)
            loss.backward()
            optimizer.step()

            losses.append(loss.item())
            all_preds.extend(outputs.detach().cpu().numpy())
            all_labels.extend(y.cpu().numpy())

        auc = roc_auc_score(all_labels, all_preds)
        print(f"Epoch {epoch+1}: Loss={np.mean(losses):.4f}, AUC={auc:.4f}")

    model_path = os.path.join(save_dir, f"{gene_name}_transformer.pt")
    torch.save(model.state_dict(), model_path)
    print(f"Saved model to {model_path}")
    
    result_df = pd.DataFrame({
        "Gene": [gene_name],
        "AUC": [auc]
    })
    result_df.to_csv(f"transformer_models/{gene_name}_results.csv", index=False)
    print(f"Saved model to {model_path}")

    return model, max_len



In [ ]:
# Constants
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

GENE_LIST = ['rpoB','rpsL','tlyA','pncA','eis','gid','katG','inhA','embA','embB', 'embC','gyrB', 'gyrA', 'ethA', 'ethR']
DATA_DIR = "./data/sequence_data_csv"
CATALOG_DF = pd.read_csv("./data/filtered_variants_output.csv")

In [ ]:
all_results = []

for gene in tqdm(GENE_LIST):
    file_path = f"./data/sequence_data_csv/{gene}_combined_sequence_data.csv"
    if os.path.exists(file_path):
        df = pd.read_csv(file_path)
        model, max_len = train_transformer_model(gene, df)

        # Optional: collect result metrics (e.g., final AUC)
        all_results.append({
            "gene": gene,
            "max_len": max_len,
            # If `train_transformer_model` prints and returns AUC, add that here too
        })
    else:
        print(f"Skipping {gene}: file not found.")

# Save summary
summary_df = pd.DataFrame(all_results)
summary_df.to_csv("transformer_models/summary_auc_scores.csv", index=False)

In [ ]:
# Helper: Load dataset and model
def load_gene_data(gene_name):
    df = pd.read_csv(f"{DATA_DIR}/{gene_name}_combined_sequence_data.csv")
    df = df[df["Frameshift_Mutation"] == 0].copy()
    df = df[df["Protein_Sequence"].apply(lambda x: x.count("X") / len(x) < 0.05)].reset_index(drop=True)
    seqs = df["Protein_Sequence"].tolist()
    labels = df["Phenotype"].map({"S": 0.0, "R": 1.0}).values
    max_len = df["Protein_Sequence"].str.len().max()
    return seqs, labels,max_len


In [ ]:

def compute_transformer_residue_importance(model, dataloader, device, max_len, save_path=None):
    """
    Compute per-residue attribution scores for a trained Transformer model using input gradient method.

    Args:
        model: Trained nn.Module (Transformer).
        dataloader: DataLoader providing batches of (input, label).
        device: 'cuda' or 'cpu'.
        max_len: Length of the input sequences (padded).
        save_path: Optional path to save importance and labels as .npz.

    Returns:
        importance_matrix: (N, max_len) numpy array of attribution scores.
        labels_array: (N,) numpy array of ground truth labels.
    """

    model.eval()
    model.to(device)
    all_importance_scores = []
    all_labels = []

    for inputs, labels in tqdm(dataloader, desc="Transformer Residue Attribution"):
        inputs = inputs.to(device)                        # shape: (B, 20, max_len)
        labels = labels.to(device)
        inputs.requires_grad_()                          # enable gradient tracking

        outputs = model(inputs)                          # shape: (B,)
        grads = []

        for i in range(len(inputs)):
            model.zero_grad()
            outputs[i].backward(retain_graph=True)

            grad = inputs.grad[i].detach().cpu().numpy()     # shape: (20, max_len)
            score = np.abs(grad).sum(axis=0)                 # → (max_len,)
            grads.append(score)

        all_importance_scores.extend(grads)
        all_labels.extend(labels.cpu().numpy())

        # Reset for next batch
        inputs.requires_grad_(False)
        inputs.grad = None

    importance_matrix = np.array(all_importance_scores)     # shape: (N, max_len)
    labels_array = np.array(all_labels)

    if save_path:
        np.savez(save_path, importance=importance_matrix, labels=labels_array)
        print(f"Saved attribution results to: {save_path}")

    return importance_matrix, labels_array


In [ ]:
WHO_catalog_df = CATALOG_DF  # for clarity

all_results = []

for gene_name in GENE_LIST:
    print(f"\n===== {gene_name} =====")

    # Load sequence data and labels
    seqs, labels, max_len = load_gene_data(gene_name)
    dataset = ProteinDataset(seqs, labels, max_len)
    dataloader = DataLoader(dataset, batch_size=32, shuffle=False)

    # Load trained Transformer model
    model_path = f"trained_transformer_models/{gene_name}_transformer.pt"
    model = ProteinTransformer(input_dim=20, max_len=max_len).to(DEVICE)
    model.load_state_dict(torch.load(model_path))

    # Compute input-gradient based importance
    imp_matrix, labels_arr = compute_transformer_residue_importance(
        model, dataloader, device=DEVICE, max_len=max_len,
        save_path=f"residue_importance/{gene_name}_transformer_residue_importance_global.npz"
    )


In [14]:
import numpy as np
import pandas as pd
from pathlib import Path

# -------------------------------
# Constants and setup
# -------------------------------
GENE_LIST = ['rpoB', 'rpsL', 'tlyA', 'pncA', 'eis', 'gid', 'katG', 'inhA', 'embA', 'embB', 'embC', 'gyrB', 'gyrA', 'ethA', 'ethR']
K_VALUES = [10, 20, 30, 50, 100]
ALLOWED_CONFIDENCES = ['1) Assoc w R', '2) Assoc w R - Interim']
IMPORTANCE_DIR = Path("residue_importance/transformer")
CATALOG_PATH = Path("./data/filtered_variants_output.csv")
OUTPUT_PATH = Path("residue_importance/transformer_pr_overlap_summary_intersectional_only.csv")

# -------------------------------
# Functions
# -------------------------------
def load_catalog(catalog_path, allowed_confidences):
    catalog = pd.read_csv(catalog_path)
    catalog = catalog[
        (catalog["confidence"].isin(allowed_confidences)) &
        (catalog["intersectional"] == True)
    ].copy()
    catalog["aa_pos_0idx"] = catalog["aa_pos"].astype(int) - 1
    return catalog

def compute_overlap(catalog_df, topk_df):
    important_positions = set(topk_df["Residue_Position"])
    overlap = catalog_df[catalog_df["aa_pos_0idx"].isin(important_positions)]
    overlap_sorted = overlap.merge(topk_df, left_on="aa_pos_0idx", right_on="Residue_Position")
    overlap_sorted = overlap_sorted.sort_values("Importance", ascending=False)
    tp_positions = overlap_sorted["aa_pos_0idx"].unique()
    tp = len(tp_positions)
    identified_variants = ", ".join(overlap_sorted.drop_duplicates("aa_pos_0idx")["variant"])
    return tp, identified_variants

def evaluate_gene(gene_name, catalog_df):
    results = []
    
    # Load per-isolate importance scores
    imp_path = IMPORTANCE_DIR / f"{gene_name}_transformer_residue_importance_global.npz"
    if not imp_path.exists():
        print(f"Skipping {gene_name}: missing importance file.")
        return results

    z = np.load(imp_path)
    imp_matrix = z["importance"]  # shape: (N isolates, max_len residues)
    mean_importance = imp_matrix.mean(axis=0)  # (max_len,)

    imp_df = pd.DataFrame({
        "Residue_Position": np.arange(len(mean_importance)),
        "Importance": mean_importance
    })

    # Subset WHO catalog to current gene
    gene_catalog = catalog_df[catalog_df["gene"].str.lower() == gene_name.lower()].copy()
    if gene_catalog.empty:
        print(f"Skipping {gene_name}: no intersectional variants found.")
        return results

    total_positives = len(set(gene_catalog["aa_pos_0idx"]))

    for k in K_VALUES:
        topk_df = imp_df.nlargest(k, "Importance")
        tp, identified_variants = compute_overlap(gene_catalog, topk_df)

        precision = tp / k if k > 0 else 0
        recall = tp / total_positives if total_positives > 0 else 0
        f1 = (2 * precision * recall) / (precision + recall + 1e-8)

        results.append({
            "Gene": gene_name,
            "Total_Resistance_Positions": total_positives,
            "Percentile": k,
            "True_Positives": tp,
            "Precision": precision,
            "Recall": recall,
            "F1_Score": f1,
            "Identified_Variants": identified_variants
        })

    return results

# -------------------------------
# Main script
# -------------------------------

catalog_df = load_catalog(CATALOG_PATH, ALLOWED_CONFIDENCES)
all_summaries = []

for gene_name in GENE_LIST:
    gene_results = evaluate_gene(gene_name, catalog_df)
    all_summaries.extend(gene_results)

final_df = pd.DataFrame(all_summaries)
final_df.to_csv(OUTPUT_PATH, index=False)
print(f"Saved summary to {OUTPUT_PATH}")


Skipping eis: no intersectional variants found.
Skipping embA: no intersectional variants found.
Skipping embC: no intersectional variants found.
Skipping ethR: no intersectional variants found.
Saved summary to residue_importance/transformer_pr_overlap_summary_intersectional_only.csv


In [16]:
# Sort by 'gene' ascending and 'precision' descending
df_sorted = final_df.sort_values(by=['Gene', 'Precision'], ascending=[True, False])

# Now, for each gene, pick the first row (highest precision per gene)
best_per_gene = df_sorted.drop_duplicates(subset=['Gene'], keep='first').reset_index(drop=True)

print(best_per_gene)

    Gene  Total_Resistance_Positions  Percentile  True_Positives  Precision  \
0   embB                           6          10               1       0.10   
1   ethA                           9         100               3       0.03   
2    gid                           8          10               1       0.10   
3   gyrA                           5          50               1       0.02   
4   gyrB                           5          10               0       0.00   
5   inhA                           1          10               0       0.00   
6   katG                           2          10               1       0.10   
7   pncA                          95          20              13       0.65   
8   rpoB                          26          10               1       0.10   
9   rpsL                           2          10               2       0.20   
10  tlyA                           2          10               0       0.00   

      Recall  F1_Score                             

In [ ]:
best_per_gene.to_csv("residue_importance/transformer_pr_overlap_summary.csv", index=False)